In [1]:
pip install torch torchvision nltk pypdf -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pypdf -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pypdf import PdfReader

#chemin="/content/code_travail_ivoirien.pdf"
reader = PdfReader("corpus/code_travail_ivoirien.pdf")
print(f"Nombre de pages : {len(reader.pages)}")

# On extrait quelques textes pour voir comme ils s'affichent
for num in [3, 50, 260, 500]:
    texte = reader.pages[num].extract_text()
    print(f"\n{'='*60}\nPAGE {num + 1}\n{'='*60}")
    print(texte[:800] if texte else "AUCUN TEXTE EXTRAIT")

Nombre de pages : 516

PAGE 4
5  
III- AGENCE EMPLOI JEUNES................................ ................................ ................................ . 410 
LA CONVENTION COLLECTIVE  INTERPROFESSIONNELLE ................................ ................ 427 
CLAUSES GENERALES ................................ ................................ ................................ ........... 428 
TITRE PREMIER : ................................ ................................ ................................ .................... 429 
DISPOSITIONS GENERALES ................................ ................................ ................................ ... 429 
TITRE II : ................................ ................................ ................................ ................................ . 435 
EXERCIC

PAGE 51
5
2 
 
En cas de licenciement, le jugement doit mentionner expressément le motif 
allégué par l'employeur.  
Le montant des dommages et intérêts est fixé en ten

PIEGE RENCONTRE — Avec pypdf, le texte extrait contient des espaces au milieu des mots : « r ésiliation
», « dur ée », « maxim ale » — surtout autour des lettres accentuées. Fatal pour la suite : « r ésiliation » ne
matchera jamais une recherche sur « résiliation », ni côté lexical ni côté embeddings. Avant de penser à un prétraitement spécifique, nous allons utiliser d'autre bibliothèques python d'extraction de pdf

In [4]:
pip install pymupdf pdfplumber -q

Note: you may need to restart the kernel to use updated packages.


In [5]:
import fitz  # PyMuPDF
import pdfplumber

PAGE_TEST = 260  # la page du décret, avec ses "r ésiliation"

# --- PyMuPDF ---
doc = fitz.open("corpus/code_travail_ivoirien.pdf")
print("=" * 25, "PyMuPDF", "=" * 25)
print(doc[PAGE_TEST].get_text()[:800]) # 800 signifie qu'on prenque 800 caractères

# pdfplumber
with pdfplumber.open("corpus/code_travail_ivoirien.pdf") as pdf:
    print("=" * 25, "pdfplumber", "=" * 25)
    print(pdf.pages[PAGE_TEST].extract_text()[:800])

========================= PyMuPDF =========================
2
 
DECRET N° 96-201 DU 7 MARS 1996 RELATIF A L'INDEMNITE DE LICENCIEMENT 
 
ARTICLE PREMIER 
La résiliation du contrat de travail du fait de l'employeur entraîne, pour le 
travailleur ayant accompli une durée de service effectif égale à un (1) an et qui n'a 
pas commis de faute lourde, le paiement d'une indemnité de licenciement distincte 
du préavis. 
 
ARTICLE 2 
Le travailleur qui a atteint la durée de service prévue ci-dessus est admis au 
bénéfice de l'indemnité de licenciement à la suite de plusieurs embauches dans la 
même entreprise, si ses départs précédents ont été provoqués par une suppression 
d'emploi ou une compression d'effectifs. 
Dans ce cas, le montant de l'indemnité est déterminé, déduction faite des sommes 
qui ont été versées à ce titre lors des licenciements antérieu
========================= pdfplumber =========================
DECRET N° 96-201 DU 7 MARS 1996 RELATIF A L'INDEMNITE DE LICENCIEMENT
ARTICL

tout fonction correctement avec l'extracteur pdfplumber. On va enchainer cela en un pipeline

In [6]:
# pipeline
import pdfplumber
import re
import json

PDF = "corpus/code_travail_ivoirien.pdf"

# Les frontières des grandes sections (pages PDF, index 0)
# On divise tout le document en des sections pour mieux orienter notre IA par la suite
SECTIONS = {
    "loi":  (6, 194),
    "decrets":    (201, 407),
    "agence_emploi_jeunes": (408, 424),
    "convention": (425, 515),
}

def nettoyer(texte):
    if not texte:
        return ""
    lignes = texte.split("\n")
    propres = []
    for l in lignes:
        l = l.strip()
        if re.fullmatch(r"[\d\s]*", l):          # folio résiduel ou ligne vide
            continue
        if "......" in l:                         # pointillés de sommaire
            continue
        propres.append(l)
    texte = "\n".join(propres)
    texte = re.sub(r"(\w)-\n(\w)", r"\1\2", texte)   # recolle les mots coupés en fin de ligne
    texte = re.sub(r"[ \t]+", " ", texte)             # espaces multiples → simple
    return texte

with pdfplumber.open(PDF) as pdf:
    for nom, (debut, fin) in SECTIONS.items():
        pages = []
        for i in range(debut, fin + 1):
            t = pdf.pages[i].extract_text()
            pages.append(nettoyer(t))
        contenu = "\n".join(pages)
        with open(f"corpus/extraits/{nom}.txt", "w", encoding="utf-8") as f:
            f.write(contenu)
        print(f"{nom:12s} : pages {debut}-{fin}, {len(contenu):,} caractères")

loi          : pages 6-194, 212,284 caractères
decrets      : pages 201-407, 216,952 caractères
agence_emploi_jeunes : pages 408-424, 18,182 caractères
convention   : pages 425-515, 121,537 caractères


Ce que fait ce code ci-dessus (extraction + nettoyage) : ouvrir le PDF avec pdfplumber, découper le fichier en 4 grandes zones géographiques (loi / décrets / agence / convention), nettoyer chaque page (folios, pointillés, mots coupés, espaces multiples), et écrire 4 gros fichiers texte sur le disque — un par section, chacun contenant tout le texte de la zone d'un seul tenant. À la sortie, On a 4 fichiers .txt d'environ 200 000 caractères chacun. Aucune notion d'article, aucune métadonnée, aucun découpage sémantique.

# Chuncking

PHASE 2 — CHUNKING STRUCTUREL v3
Pourquoi cette étape : transformer les 4 fichiers texte extraits du PDF en
"chunks" = un article par chunk, avec ses métadonnées (section, statut
juridique, texte parent, numéro d'article). C'est la granularité idéale pour
un RAG juridique : assez fin pour un retrieval précis, assez complet pour
porter le sens, et chaque chunk est citable ("Art. 2, Décret 96-201...").

In [7]:
"""
Ceci est la version 3 du chunkings. Les versions 1 et 2 ont été supprimer dans le but d'airer le code et supprimer tout confusion.
Changements v3 (après diagnostic des titres tronqués/avalés) :
  1. RE_TEXTE_PARENT reconnaît DECRET, ARRETE et ORDONNANCE comme textes
     parents (le corpus contient les trois types — un article d'arrêté doit
     être cité "Art. X, Arrêté n°2250...", pas rester orphelin).
  2. Les lignes de continuation d'un titre excluent tout début de nouveau
     texte (DECRET/ARRETE/ORDONNANCE/ANNEXE), les en-têtes d'articles
     (ARTICLE/Art.) et les têtes de structure (TITRE/CHAPITRE) — c'est ce
     qui empêchait la v2 d'avaler des blocs entiers de titres consécutifs.
  3. Prérequis : decrets.txt régénéré SANS le sommaire (extraction avec
     SECTIONS["decrets"] = (201, 407)).

Sortie : corpus/chunks.json.
Contrôle de succès : ~528 chunks décrets, et le parent de l'article 8 du
décret 96-195 doit afficher le titre COMPLET ("...DUREE DE LA PERIODE D'ESSAI").
"""

from pathlib import Path
import re
import json

# Chemins
RACINE = Path(r"C:\Users\KONAN GERVAIS\Desktop\juriprudence\corpus")
EXTRAITS = RACINE / "extraits"
SORTIE = RACINE / "chunks.json"

#  En-tête d'article, tous formats confondus, en début de ligne
RE_ARTICLE = re.compile(
    r"^(?:Art\.|Article|ARTICLE)\s+(\d+(?:\.\d+)?|PREMIER|premier)\s*",
    re.MULTILINE,
)

# En-tête de texte parent (décret, arrêté, ordonnance)
# STOP : mots qui ne peuvent PAS ouvrir une ligne de continuation de titre.
STOP = r"(?!ARTICLE|Article|Art\.|DECRET|ARRETE|ORDONNANCE|ANNEXE|TITRE|CHAPITRE)"

RE_TEXTE_PARENT = re.compile(
    r"^((?:DECRET|ARRETE|ORDONNANCE)\s+N°\s*[^\n]+"    # 1ère ligne du titre
    rf"(?:\n{STOP}[A-ZÉÈÀÇÔÎ0-9][^\n]*)*)",             # continuations éventuelles
    re.MULTILINE,
)

STATUTS = {
    "loi": "Loi n°2015-532 du 20/07/2015 (Code du travail)",
    "decrets": "Décret d'application",
    "agence_emploi_jeunes": "Textes Agence Emploi Jeunes",
    "convention": "Convention collective interprofessionnelle du 19/07/1977",
}


def chunker_section(nom, texte):
    """Découpe le texte d'une section en chunks, un par article.

    Chaque article est rattaché à son texte parent (décret, arrêté ou
    ordonnance) via les positions : un article appartient au dernier texte
    parent qui commence avant lui dans le fichier. Indispensable pour les
    décrets, dont la numérotation d'articles recommence à 1 dans chaque texte.

    Args:
        nom: clé de la section ("loi", "decrets", "agence_emploi_jeunes",
             "convention") — alimente les métadonnées.
        texte: contenu complet du fichier .txt de la section.

    Returns:
        Liste de dicts : id, section, statut, texte_parent (titre complet
        ou None), article (numéro), contenu, nb_caracteres.
    """
    chunks = []

    #  Repérer les textes parents et leur position de début.
    #    re.sub remplace les sauts de ligne internes au titre par une espace,
    #    pour stocker un titre propre sur une seule ligne.
    parents = [
        (m.start(), re.sub(r"\s*\n\s*", " ", m.group(1)).strip())
        for m in RE_TEXTE_PARENT.finditer(texte)
    ]

    def parent_de(pos):
        """Titre du dernier texte parent commençant avant `pos` (ou None)."""
        courant = None
        for debut, titre in parents:
            if debut <= pos:
                courant = titre
            else:
                break
        return courant

    #  Découper aux en-têtes d'articles : chaque chunk va d'un en-tête au
    #    début de l'en-tête suivant (ou à la fin du texte pour le dernier).
    matches = list(RE_ARTICLE.finditer(texte))
    for i, m in enumerate(matches):
        debut = m.start()
        fin = matches[i + 1].start() if i + 1 < len(matches) else len(texte)
        contenu = texte[debut:fin].strip()
        numero = m.group(1)
        if numero.upper() == "PREMIER":
            numero = "1"
        chunks.append({
            "id": f"{nom}_{i:04d}",
            "section": nom,
            "statut": STATUTS[nom],
            "texte_parent": parent_de(debut),
            "article": numero,
            "contenu": contenu,
            "nb_caracteres": len(contenu),
        })
    return chunks


# Exécution
tous = []
for nom in STATUTS:
    texte = (EXTRAITS / f"{nom}.txt").read_text(encoding="utf-8")
    chunks = chunker_section(nom, texte)
    tous.extend(chunks)
    tailles = [c["nb_caracteres"] for c in chunks]
    print(f"{nom:22s} : {len(chunks):4d} chunks | "
          f"taille min/méd/max : {min(tailles)}/{sorted(tailles)[len(tailles)//2]}/{max(tailles)}")

SORTIE.write_text(json.dumps(tous, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n→ {len(tous)} chunks écrits dans {SORTIE}")

loi                    :  406 chunks | taille min/méd/max : 70/370/2424
decrets                :  528 chunks | taille min/méd/max : 78/327/5752
agence_emploi_jeunes   :   42 chunks | taille min/méd/max : 88/253/3278
convention             :  117 chunks | taille min/méd/max : 96/856/5055

→ 1093 chunks écrits dans C:\Users\KONAN GERVAIS\Desktop\juriprudence\corpus\chunks.json


In [8]:
"""
CONTRÔLE QUALITÉ v3 — verdict final de la Phase 2.
Pourquoi cette étape : les comptages sont stables (1093 chunks, identiques à
la v2), mais la v3 visait les MÉTADONNÉES : titres de textes parents complets
et individuels. On vérifie donc :
  1. TITRE RÉPARÉ — l'article 8 du décret 96-195 doit citer le titre COMPLET
     ("...ET A LA DUREE DE LA PERIODE D'ESSAI"), plus la version tronquée
     ("...ET A LA") ni le bloc-sommaire avalé de la v2.
  2. PANORAMA DES PARENTS — liste des textes parents distincts détectés dans
     les décrets : chacun doit être UN titre individuel et complet. C'est le
     contrôle le plus parlant : tout avalement ou troncature s'y verra.
  3. TÉMOINS — 2 chunks au hasard pour la qualité moyenne.
Critère de succès : titres complets et individuels -> Phase 2 SCELLÉE.
"""

import random
import json
from pathlib import Path

RACINE = Path(r"C:\Users\KONAN GERVAIS\Desktop\juriprudence\corpus")
SORTIE = RACINE / "chunks.json"

chunks = json.loads(SORTIE.read_text(encoding="utf-8"))

print("## VÉRIF TITRE RÉPARÉ (décret 96-195)##")
cible = next(
    c for c in chunks
    if c["section"] == "decrets"
    and c["texte_parent"] and "96-195" in c["texte_parent"]
)
print(f"parent : {cible['texte_parent']}")

print("\n########  PANORAMA DES TEXTES PARENTS (décrets) ########")
parents_distincts = sorted({
    c["texte_parent"] for c in chunks
    if c["section"] == "decrets" and c["texte_parent"]
})
print(f"{len(parents_distincts)} textes parents distincts :\n")
for p in parents_distincts:
    print(f"  • {p}")

print("\n######## 2 TÉMOINS AU HASARD ########")
for c in random.sample(chunks, 2):
    print(f"\n[{c['id']}] art. {c['article']} | {c['section']} | parent : {c['texte_parent']}")
    print(c["contenu"][:300])

######## 1. VÉRIF TITRE RÉPARÉ (décret 96-195) ########
parent : DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A LA DUREE DE LA PERIODE D'ESSAI

######## 2. PANORAMA DES TEXTES PARENTS (décrets) ########
36 textes parents distincts :

  • ARRETE N° 009 MEMEASS/CAB DU 19 JANVIER 2012 REVISANT L’ARRETE N° 2250 DU 14 MARS 2005 PORTANT DETERMINATION DE LA LISTE DES TRAVAUX DANGEREUX INTERDITS AUX ENFANTS DE MOINS DE DIX HUIT ANS
  • ARRETE N° 2015-855 DU 30 DEC 2015 PORTANT APPLICATION DU BAREME DES SALAIRES MINIMA CATEGORIELS CONVENTIONNELS DE 2015
  • ARRETE N° 2250 DU 14 MARS 2005 PORTANT DETERMINATION DE LA LISTE DES TRAVAUX DANGEREUX INTERDITS AUX ENFANTS DE MOINS DE DIX HUIT (18) ANS
  • DECRET N° 2013-554 DU 5 AOUT 2013 PORTANT ETABLISSEMENT DE LA LISTE DES MALADIES PROFESSIONNELLES INDEMNISABLES.
  • DECRET N° 2013-555 DU 5 AOUT 2013 PORTANT CREATION, ATTRIBUTIONS, ORGANISATION ET FONCTIONNEMENT DE L'OBSERVATOIRE NATIONAL DES ACCIDENTS DU TRAVAIL ET DES MALADI

Chunking structurel + métadonnées citables — Découpage du corpus juridique (516 pages : Code du travail ivoirien, décrets, convention collective, textes annexés) en 1 093 chunks, un article par unité, chacun rattaché à son texte parent pour être citable de façon exacte.
Trois pièges de métadonnées résolus successivement : titres de décrets tronqués (multi-lignes), titres avalés (sommaire résiduel absorbé), et références mensongères (lois annexées étiquetées « Code du travail » — inacceptable en droit). Remèdes : regex de capture multi-lignes avec liste d'exclusion, ajustement des frontières d'extraction à la source, champ reference honnête calculé au chunking. Validation par un contrôle à trois angles (cas connu / vue déduplicative / échantillon aléatoire).
Chiffres : 1 093 chunks, 36 textes parents distincts correctement identifiés, 0 métadonnée mensongère à l'inspection finale.

# Embedding

L'IA ne sait pas lire des textes, elle sait lire que des chiffre. L'embedding, consiste à convertir chaque texte en vecteur de grande dimension (ici 768). Les vecteurs colinéaires, auront un lien alors que ceux qui sont non colinéaire n'en auront pas.

In [9]:
pip install sentence-transformers chromadb -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
PHASE 3 — EMBEDDINGS & INDEX VECTORIEL

Pourquoi cette étape : transformer chaque chunk en vecteur dense (embedding)
qui capture son SENS, et stocker le tout dans ChromaDB, une base vectorielle
persistante. Contrairement à TF-IDF (mots exacts), les embeddings rapprochent
"congés" de "vacances", "licenciement" de "rupture du contrat" — c'est ce qui
permettra de répondre à des questions en langage naturel.

Choix techniques :
  - Modèle : intfloat/multilingual-e5-base — fort en retrieval multilingue,
    fenêtre de 512 tokens (couvre confortablement nos articles médians).
    Convention e5 : préfixe "passage: " pour les documents indexés,
    "query: " pour les questions (le modèle a été entraîné ainsi).
  - normalize_embeddings=True : vecteurs de norme 1, donc similarité cosinus
    = simple produit scalaire (exactement comme notre TF-IDF normalisé L2 ).
  - ChromaDB persistant : l'index survit aux redémarrages, on n'encode qu'UNE
    fois.

Limitation connue : les articles > ~2000 caractères sont tronqués à 512
tokens par le modèle — acceptable en V1, à réévaluer par la suite.

Sortie : corpus/index_chroma/ (base persistante, collection "droit_ivoirien").
"""

import json
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

RACINE = Path(r"C:\Users\KONAN GERVAIS\Desktop\juriprudence\corpus")
CHUNKS = RACINE / "chunks.json"
INDEX_DIR = str(RACINE / "index_chroma")

MODELE = "intfloat/multilingual-e5-base"

# 1. Charger les chunks
chunks = json.loads(CHUNKS.read_text(encoding="utf-8"))
print(f"{len(chunks)} chunks chargés")

# 2. Charger le modèle (télécharge )
modele = SentenceTransformer(MODELE)
print(f"Modèle chargé : {MODELE} | fenêtre : {modele.max_seq_length} tokens")

# 3. Encoder tous les chunks (préfixe e5 "passage: ")
textes = [f"passage: {c['contenu']}" for c in chunks]
embeddings = modele.encode(
    textes,
    batch_size=32,
    show_progress_bar=True,        # barre de progression
    normalize_embeddings=True,     # norme 1 → cosinus = produit scalaire
)
print(f"Embeddings : {embeddings.shape}")   # attendu : (1093, 768)

# 4. Créer l'index ChromaDB persistant ---
client = chromadb.PersistentClient(path=INDEX_DIR)
# On repart de zéro si la collection existe (ré-exécution propre de la cellule)
try:
    client.delete_collection("droit_ivoirien")
except Exception:
    pass
collection = client.create_collection(
    name="droit_ivoirien",
    metadata={"hnsw:space": "cosine"},   # distance cosinus pour les requêtes
)

# ChromaDB refuse None dans les métadonnées → on convertit en chaîne vide
collection.add(
    ids=[c["id"] for c in chunks],
    embeddings=embeddings.tolist(),
    documents=[c["contenu"] for c in chunks],
    metadatas=[{
        "section": c["section"],
        "statut": c["statut"],
        "texte_parent": c["texte_parent"] or "",
        "article": c["article"],
    } for c in chunks],
)
print(f"Index créé : {collection.count()} chunks dans {INDEX_DIR}")

1093 chunks chargés


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modèle chargé : intfloat/multilingual-e5-base | fenêtre : 512 tokens


Batches:   0%|          | 0/35 [00:00<?, ?it/s]

# Smoke test

In [11]:
"""
SMOKE TEST — première interrogation sémantique de l'index.
Pourquoi : avant toute évaluation formelle, vérifier à l'œil que le retrieval
"fonctionne" : une question en français courant doit remonter des articles
pertinents, même sans mots-clés exacts. Noter le préfixe "query: " (convention
e5, différent du préfixe des passages).
"""

question = "Quelle est la durée maximale de la période d'essai ?"
q_emb = modele.encode([f"query: {question}"], normalize_embeddings=True)

resultats = collection.query(query_embeddings=q_emb.tolist(), n_results=3)

print(f"Question : {question}\n")
for doc, meta, dist in zip(
    resultats["documents"][0], resultats["metadatas"][0], resultats["distances"][0]
):
    print(f"— similarité {1 - dist:.3f} | Art. {meta['article']} | {meta['texte_parent'] or meta['statut']}")
    print(f"  {doc[:200]}\n")

Question : Quelle est la durée maximale de la période d'essai ?

— similarité 0.865 | Art. 2 | DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A LA DUREE DE LA PERIODE D'ESSAI
  ARTICLE 2
Dans l'un ou l'autre cas, la durée de l'essai est fixée par écrit comme suit :
huit jours pour les travailleurs payés à l'heure ou à la journée ;
un mois pour les travailleurs payés au mois 

— similarité 0.864 | Art. 14.5 | Loi n°2015-532 du 20/07/2015 (Code du travail)
  Art. 14.5
Le contrat de travail, qu'il soit à durée déterminée ou à durée indéterminée, peur
comporter une période d'essai dont la durée totale maximale est fixée par décret.
Lorsque les parties au co

— similarité 0.856 | Art. 14 | Convention collective interprofessionnelle du 19/07/1977
  ARTICLE 14
PERIODE D'ESSAI
L'engagement définitif du travailleur peut être précédé d'une période d'essai
stipulée obligatoirement par écrit et dont la durée maximale varie selon la
catégorie professio



### Un autre SMOKE TEST

In [12]:
"""
CALIBRATION — les bons articles sont-ils dans le top 10 de la question ratée ?
Pourquoi : si les articles sur le licenciement pour faute lourde apparaissent
en positions 4-10 sur "virer sans préavis", alors k=8-10 + le tri du LLM
suffiront ; s'ils sont absents du top 10, l'hybride BM25 devra aussi porter
ce cas, pas seulement les références exactes. Cette mesure décide du poids
relatif des deux moteurs en Phase 4.
"""

question = "Mon patron peut-il me virer sans préavis ?"
q_emb = modele.encode([f"query: {question}"], normalize_embeddings=True)
res = collection.query(query_embeddings=q_emb.tolist(), n_results=10)

print(f"Q : {question} — TOP 10 :\n")
for rang, (doc, meta, dist) in enumerate(zip(
    res["documents"][0], res["metadatas"][0], res["distances"][0]
), start=1):
    source = meta["texte_parent"] or meta["statut"]
    print(f"{rang:2d}. {1 - dist:.3f} | Art. {meta['article']} | {source[:60]}")
    print(f"     {doc[:110]}")

Q : Mon patron peut-il me virer sans préavis ? — TOP 10 :

 1. 0.837 | Art. 24 | DECRET N° 96-207 DU 7 MARS 1996 RELATIF AUX DELEGUES DU PERS
     ARTICLE 24
Les circonstances exceptionnelles supprimant l’obligation du préavis de deux
jours pour la réceptio
 2. 0.837 | Art. 27 | Convention collective interprofessionnelle du 19/07/1977
     ARTICLE 27
MISE EN DISPONIBILITE
Le travailleur peut bénéficier, sur sa demande, d'une mise en disponibilité s
 3. 0.832 | Art. 13 | DECRET N° 96-198 DU 7 MARS 1996 RELATIF AUX CONDITIONS DE SU
     ARTICLE 13
L'employeur a l'obligation de recevoir le travailleur dont le contrat a été suspendu
pour cause de 
 4. 0.832 | Art. 11.9 | Loi n°2015-532 du 20/07/2015 (Code du travail)
     Art. 11.9
Le nouvel employeur garde néanmoins le droit de procéder à des ruptures de
contrat de travail dans l
 5. 0.832 | Art. 17 | Convention collective interprofessionnelle du 19/07/1977
     ARTICLE 17
PROMOTION
L'employeur est tenu de faire appel de préférence aux tr

PHASE 4a — RÉ-INDEXATION ENRICHIE + INDEX BM25

Pourquoi cette étape (verdicts de la calibration Phase 3) :
  1. "article 14.5" ne remontait pas l'article 14.5 : les embeddings encodent
     mal les identifiants exacts. Remède ici : ENRICHIR chaque passage encodé
     avec sa référence ("Code du travail, Art. 14.5 — ...") pour que la
     référence soit DANS le texte vu par les deux moteurs.
  2. Le retrieval dense échoue totalement sur certaines questions ("virer
     sans préavis" : 0 pertinent dans le top 10). Remède : un second moteur,
     BM25 (lexical, l'héritier industriel de TF-IDF), fusionné en Phase 4b.
Ce que fait cette cellule :
  - reconstruit la collection ChromaDB avec les passages enrichis ;
  - construit en parallèle un index BM25 sur les mêmes textes enrichis ;
  - sauvegarde l'ordre des ids (commun aux deux moteurs, pour la fusion).

In [13]:
pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [14]:

import json
import re
from pathlib import Path

import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

RACINE = Path(r"C:\Users\KONAN GERVAIS\Desktop\juriprudence\corpus")
CHUNKS = RACINE / "chunks.json"
INDEX_DIR = str(RACINE / "index_chroma")

MODELE = "intfloat/multilingual-e5-base"

REF_COURTES = {
    "loi": "Code du travail",
    "decrets": None,          # remplacé par le titre du décret parent
    "agence_emploi_jeunes": "Textes Agence Emploi Jeunes",
    "convention": "Convention collective interprofessionnelle",
}


def texte_enrichi(c):
    """Construit le passage enrichi : référence lisible + contenu.

    Pourquoi : la référence ("Code du travail, Art. 14.5") devient du texte
    indexable — le dense y gagne du contexte, BM25 peut matcher "14.5"
    littéralement. C'est LA correction du test "article 14.5" raté.

    Args:
        c: un chunk (dict) de chunks.json.

    Returns:
        str: "référence — contenu" prêt à encoder/indexer.
    """
    ref = REF_COURTES[c["section"]] or c["texte_parent"] or c["statut"]
    return f"{ref}, Art. {c['article']} — {c['contenu']}"


def tokeniser(texte):
    """Tokenisation simple pour BM25 : minuscules + mots alphanumériques.

    Pourquoi si simple : BM25 n'a besoin que d'un découpage cohérent entre
    l'index et les requêtes. \\w+ garde les nombres ("14.5" devient "14" "5",
    deux tokens consécutifs — suffisant pour matcher la référence).
    """
    return re.findall(r"\w+", texte.lower())


# 1. Charger chunks et modèle
chunks = json.loads(CHUNKS.read_text(encoding="utf-8"))
modele = SentenceTransformer(MODELE)
passages = [texte_enrichi(c) for c in chunks]
print(f"{len(passages)} passages enrichis — exemple :\n{passages[60][:150]}\n")

# 2. Ré-encoder (dense) et reconstruire ChromaDB
embeddings = modele.encode(
    [f"passage: {p}" for p in passages],
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

client = chromadb.PersistentClient(path=INDEX_DIR)
try:
    client.delete_collection("droit_ivoirien")
except Exception:
    pass
collection = client.create_collection(
    name="droit_ivoirien", metadata={"hnsw:space": "cosine"}
)
collection.add(
    ids=[c["id"] for c in chunks],
    embeddings=embeddings.tolist(),
    documents=passages,                      # on stocke la version enrichie
    metadatas=[{
        "section": c["section"],
        "statut": c["statut"],
        "texte_parent": c["texte_parent"] or "",
        "article": c["article"],
    } for c in chunks],
)
print(f"ChromaDB reconstruit : {collection.count()} passages enrichis")

# 3. Index BM25 sur les mêmes passages
bm25 = BM25Okapi([tokeniser(p) for p in passages])
ids_ordre = [c["id"] for c in chunks]        # position i ↔ ids_ordre[i]
print("Index BM25 construit (en mémoire)")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1002.23it/s]


1093 passages enrichis — exemple :
Code du travail, Art. 15.1 — Art. 15.1
Le contrat de travail à durée déterminée est un contrat qui prend fin à l'arrivée d'un
terme fixé par les parti



Batches: 100%|██████████| 35/35 [07:42<00:00, 13.21s/it]


ChromaDB reconstruit : 1093 passages enrichis
Index BM25 construit (en mémoire)



PHASE 4b — RECHERCHE HYBRIDE (dense + BM25, fusion RRF)

Pourquoi RRF (Reciprocal Rank Fusion) : les scores dense (cosinus environ 0.83) et
BM25 (environ 5-20) vivent sur des échelles incomparables — les additionner n'aurait
aucun sens. RRF ne fusionne que les RANGS : score(doc) = Σ 1/(K + rang_moteur).
Un document bien classé par les DEUX moteurs domine ; un document fort dans un
seul moteur reste visible. K=60 est la constante standard de la littérature.

Prérequis (cellule 4a exécutée dans la même session) : `modele`, `collection`,
`bm25`, `ids_ordre` et `tokeniser` doivent exister en mémoire.

Test de la revanche : les 3 questions problématiques de la Phase 3.
Succès attendu :
  - "article 14.5" → l'article 14.5 en TÊTE (grâce à BM25 + enrichissement) ;
  - "virer sans préavis" → amélioration partielle au mieux ("virer" absent du
    corpus — la reformulation LLM de la Phase 5 finira le travail) ;
  - "TVA" → toujours du hors-sujet en tête (le "je ne sais pas" reste du
    ressort du LLM, décision déjà actée en Phase 3).


In [15]:


def recherche_hybride(question, k=8, k_moteur=20, K_rrf=60):
    """Interroge dense + BM25 et fusionne les classements par RRF.

    Pourquoi cette fonction : c'est LE point d'entrée du retrieval du projet.
    Le moteur dense comprend les reformulations ("femme enceinte" →
    protection maternité), BM25 attrape les termes exacts ("article 14.5",
    "SMIG") — la fusion RRF combine leurs forces sans régler d'hyperparamètre
    d'échelle.

    Args:
        question: question en langage naturel.
        k: nombre de passages retournés après fusion.
        k_moteur: profondeur de chaque moteur avant fusion (plus large que k
                  pour laisser la fusion repêcher les accords entre moteurs).
        K_rrf: constante RRF (60 = standard, amortit le poids des premiers rangs).

    Returns:
        Liste de tuples (id, score_rrf, passage, metadata), triée par
        score décroissant.
    """
    # Moteur dense (préfixe "query: ", convention e5)
    q_emb = modele.encode([f"query: {question}"], normalize_embeddings=True)
    res = collection.query(query_embeddings=q_emb.tolist(), n_results=k_moteur)
    rangs_dense = {id_: rang for rang, id_ in enumerate(res["ids"][0])}

    #  Moteur BM25 (mêmes passages, tokenisation identique à l'index)
    scores_bm25 = bm25.get_scores(tokeniser(question))
    top_bm25 = sorted(range(len(scores_bm25)), key=lambda i: -scores_bm25[i])[:k_moteur]
    rangs_bm25 = {ids_ordre[i]: rang for rang, i in enumerate(top_bm25)}

    # Fusion RRF : somme des 1/(K + rang) sur les moteurs où le doc figure
    candidats = set(rangs_dense) | set(rangs_bm25)
    scores = {
        id_: sum(1 / (K_rrf + r[id_]) for r in (rangs_dense, rangs_bm25) if id_ in r)
        for id_ in candidats
    }
    tops = sorted(scores, key=scores.get, reverse=True)[:k]

    # --- Récupérer passages + métadonnées depuis ChromaDB ---
    docs = collection.get(ids=tops)
    par_id = {i: (d, m) for i, d, m in zip(docs["ids"], docs["documents"], docs["metadatas"])}
    return [(id_, scores[id_], par_id[id_][0], par_id[id_][1]) for id_ in tops]


# La revanche : les 3 questions problématiques de la Phase 3
questions = [
    "Que dit l'article 14.5 du Code du travail ?",
    "Mon patron peut-il me virer sans préavis ?",
    "Quel est le taux de TVA en Côte d'Ivoire ?",
]
for q in questions:
    print(f"\n{'='*70}\nQ : {q}")
    for id_, s, doc, meta in recherche_hybride(q, k=5):
        print(f"  {s:.4f} | Art. {meta['article']} | {(meta['texte_parent'] or meta['statut'])[:55]}")
        print(f"         {doc[:110]}")


Q : Que dit l'article 14.5 du Code du travail ?
  0.0316 | Art. 14.4 | Loi n°2015-532 du 20/07/2015 (Code du travail)
         Code du travail, Art. 14.4 — Art. 14.4
L'existence du contrat de travail se prouve par tous moyens.
  0.0286 | Art. 14.3 | Loi n°2015-532 du 20/07/2015 (Code du travail)
         Code du travail, Art. 14.3 — Art. 14.3
Le contrat de travail peut être conclu pour une durée indéterminée ou p
  0.0167 | Art. 14 | Loi n°2015-532 du 20/07/2015 (Code du travail)
         Code du travail, Art. 14 — ARTICLE 14 nouveau
L'esclavage est l'état ou la condition d'un enfant sur lequel s'
  0.0164 | Art. 5 | DECRET N° 95-542 DU 14 JUILLET 1995 RELATIF A LA COMPOS
         DECRET N° 95-542 DU 14 JUILLET 1995 RELATIF A LA COMPOSITION ET A LA DUREE DU MANDAT DES MEMBRES DE LA COMMISS
  0.0164 | Art. 14.1 | Loi n°2015-532 du 20/07/2015 (Code du travail)
         Code du travail, Art. 14.1 — Art. 14.1
Le contrat de travail est un accord de volontés par lequel une personne

Q : Mon


PHASE 4c — TOKENISATION RÉPARÉE + ROUTEUR DE RÉFÉRENCES EXACTES

Pourquoi (diagnostic de la revanche 4b) : "article 14.5" échoue encore, pour
trois raisons identifiées :
  1. tokeniser() découpait "14.5" en "14" + "5", deux tokens ultra-banals →
     la référence perdait sa rareté, donc son pouvoir discriminant BM25 ;
  2. BM25 favorise les documents courts (l'Art. 14.4, 70 caractères, gagnait
     mécaniquement) ;
  3. la question dit "article", la loi écrit "Art." — tokens différents.
Remèdes :
  A. Tokenisation qui préserve les nombres pointés : "14.5" devient UN token
     rare → reconstruction de l'index BM25 (rapide, en mémoire).
  B. ROUTEUR : une référence exacte n'est pas une question sémantique, c'est
     une requête de base de données. Si la question contient "article X(.Y)",
     on consulte directement les métadonnées `article` de ChromaDB et on
     PRÉPEND les résultats exacts aux résultats hybrides.
Leçon d'architecture : à chaque type de requête son moteur — sémantique pour
les reformulations, lexical pour les termes exacts, lookup pour les références.


In [16]:


import re


def tokeniser(texte):
    """Tokenise pour BM25 en préservant les nombres pointés.

    Pourquoi : "14.5" doit rester UN token (rare → fort IDF → discriminant),
    au lieu d'être éclaté en "14" + "5" (banals). L'alternance essaie d'abord
    le motif nombre-pointé, puis les mots alphanumériques classiques.

    Args:
        texte: chaîne à tokeniser (passage ou question).

    Returns:
        Liste de tokens en minuscules.
    """
    return re.findall(r"\d+\.\d+|\w+", texte.lower())


# --- Reconstruction de l'index BM25 avec la nouvelle tokenisation ---
# (les `passages` et `ids_ordre` de la cellule 4a sont réutilisés tels quels)
bm25 = BM25Okapi([tokeniser(p) for p in passages])
print("Index BM25 reconstruit avec tokenisation nombres-pointés")

# --- Routeur de références exactes ---
RE_REF_QUESTION = re.compile(r"article\s+(\d+(?:\.\d+)?)", re.IGNORECASE)


def recherche(question, k=8):
    """Point d'entrée du retrieval : routeur + recherche hybride.

    Pourquoi : si la question cite une référence exacte ("article 14.5"),
    la consultation par métadonnées est infaillible là où le fuzzy matching
    (dense comme lexical) reste faillible. Les correspondances exactes sont
    PRÉPENDUES aux résultats hybrides — le reste du top-k garde le contexte
    (articles voisins, textes d'application) utile au LLM en Phase 5.

    Args:
        question: question en langage naturel.
        k: nombre total de passages retournés.

    Returns:
        Liste de tuples (id, score, passage, metadata) — les correspondances
        exactes d'abord (score arbitraire 1.0), puis l'hybride.
    """
    resultats = []
    deja = set()

    m = RE_REF_QUESTION.search(question)
    if m:
        exacts = collection.get(where={"article": m.group(1)})
        for id_, doc, meta in zip(exacts["ids"], exacts["documents"], exacts["metadatas"]):
            resultats.append((id_, 1.0, doc, meta))
            deja.add(id_)

    for id_, s, doc, meta in recherche_hybride(question, k=k):
        if id_ not in deja and len(resultats) < k:
            resultats.append((id_, s, doc, meta))

    return resultats


# --- Contre-attaque : les mêmes questions, plus un test du routeur ---
questions = [
    "Que dit l'article 14.5 du Code du travail ?",
    "Mon patron peut-il me virer sans préavis ?",
    "Que prévoit l'article 2 sur la période d'essai ?",   # référence AMBIGUË : plusieurs "Art. 2" !
]
for q in questions:
    print(f"\n{'='*70}\nQ : {q}")
    for id_, s, doc, meta in recherche(q, k=5):
        marque = "🎯" if s == 1.0 else "  "
        print(f"{marque}{s:.4f} | Art. {meta['article']} | {(meta['texte_parent'] or meta['statut'])[:55]}")
        print(f"         {doc[:110]}")

Index BM25 reconstruit avec tokenisation nombres-pointés

Q : Que dit l'article 14.5 du Code du travail ?
🎯1.0000 | Art. 14.5 | Loi n°2015-532 du 20/07/2015 (Code du travail)
         Code du travail, Art. 14.5 — Art. 14.5
Le contrat de travail, qu'il soit à durée déterminée ou à durée indéter
  0.0167 | Art. 12 | DECRET N° 98-41 DU 28 JANVIER 1998 RELATIF AUX CONVENTI
         DECRET N° 98-41 DU 28 JANVIER 1998 RELATIF AUX CONVENTIONS COLLECTIVES DE TRAVAIL SECTION 1 : CONDITIONS DE PU
  0.0167 | Art. 14.4 | Loi n°2015-532 du 20/07/2015 (Code du travail)
         Code du travail, Art. 14.4 — Art. 14.4
L'existence du contrat de travail se prouve par tous moyens.
  0.0164 | Art. 7 | Convention collective interprofessionnelle du 19/07/197
         Convention collective interprofessionnelle, Art. 7 — ARTICLE 7
L'indemnité de cessation des relations de trava
  0.0164 | Art. 14.1 | Loi n°2015-532 du 20/07/2015 (Code du travail)
         Code du travail, Art. 14.1 — Art. 14.1
Le contrat de t

In [17]:
"""
PHASE 4d — ROUTEUR v2 : DÉSAMBIGUÏSATION DES RÉFÉRENCES AMBIGUËS
Pourquoi (verdict du test piège "article 2") : 44 textes du corpus ont un
"Art. 2" — le routeur v1 les prépendait TOUS, ignorant k et noyant la réponse.
Remèdes :
  1. FILTRE DE SECTION : si la question précise "du Code du travail", on
     restreint la consultation aux chunks de la section "loi" (idem
     "convention" → section convention). Le contexte nommé est un filtre.
  2. DÉSAMBIGUÏSATION SÉMANTIQUE : s'il reste plusieurs correspondances
     exactes, on les re-classe par similarité dense avec la QUESTION ENTIÈRE
     ("...sur la période d'essai" doit faire remonter l'Art. 2 du décret
     96-195) et on n'en garde que max_exacts. Les embeddings sont relus
     depuis ChromaDB (pas de ré-encodage des passages → rapide).
  3. Le plafond k est enfin respecté.
"""

import numpy as np

RE_REF_QUESTION = re.compile(r"article\s+(\d+(?:\.\d+)?)", re.IGNORECASE)

# Indices textuels → filtre de section
INDICES_SECTION = {
    "code du travail": "loi",
    "convention collective": "convention",
    "convention": "convention",
}


def recherche(question, k=8, max_exacts=3):
    """Point d'entrée du retrieval : routeur désambiguïsé + hybride.

    Args:
        question: question en langage naturel.
        k: nombre total de passages retournés (plafond STRICT cette fois).
        max_exacts: nombre max de correspondances exactes prépendées après
                    désambiguïsation sémantique.

    Returns:
        Liste de tuples (id, score, passage, metadata) — exacts d'abord
        (score 1.0), puis hybride, k éléments au plus.
    """
    resultats, deja = [], set()
    m = RE_REF_QUESTION.search(question)

    if m:
        # --- Filtre de section si la question nomme le texte ---
        q_low = question.lower()
        section = next((s for indice, s in INDICES_SECTION.items() if indice in q_low), None)
        if section:
            where = {"$and": [{"article": {"$eq": m.group(1)}},
                              {"section": {"$eq": section}}]}
        else:
            where = {"article": {"$eq": m.group(1)}}

        exacts = collection.get(where=where, include=["documents", "metadatas", "embeddings"])
        n = len(exacts["ids"])

        if n > 0:
            if n > max_exacts:
                # --- Désambiguïsation : re-classement par similarité dense ---
                q_emb = modele.encode([f"query: {question}"], normalize_embeddings=True)[0]
                sims = np.array(exacts["embeddings"]) @ q_emb
                ordre = np.argsort(-sims)[:max_exacts]
            else:
                ordre = range(n)
            for i in ordre:
                resultats.append((exacts["ids"][i], 1.0,
                                  exacts["documents"][i], exacts["metadatas"][i]))
                deja.add(exacts["ids"][i])

    for id_, s, doc, meta in recherche_hybride(question, k=k):
        if id_ not in deja and len(resultats) < k:
            resultats.append((id_, s, doc, meta))

    return resultats[:k]


# --- Tests : l'inondation doit être contenue, les victoires préservées ---
questions = [
    "Que prévoit l'article 2 sur la période d'essai ?",       # ambigu → désambiguïsation
    "Que dit l'article 14.5 du Code du travail ?",            # filtre section "loi"
    "Que dit l'article 2 de la convention collective ?",      # filtre section "convention"
]
for q in questions:
    print(f"\n{'='*70}\nQ : {q}")
    for id_, s, doc, meta in recherche(q, k=5):
        marque = "🎯" if s == 1.0 else "  "
        print(f"{marque}{s:.4f} | Art. {meta['article']} | {(meta['texte_parent'] or meta['statut'])[:55]}")
        print(f"         {doc[:110]}")


Q : Que prévoit l'article 2 sur la période d'essai ?
🎯1.0000 | Art. 2 | DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT 
         DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A LA DUREE DE LA PERIODE D'ESSAI, Art. 2 —
🎯1.0000 | Art. 2 | Loi n°2015-532 du 20/07/2015 (Code du travail)
         Code du travail, Art. 2 — ARTICLE 2
Les dispositions de la présente loi visent tous les enfants, quels que soi
🎯1.0000 | Art. 2 | DECRET N° 96-203 DU 7 MARS 1996 RELATIF A LA DUREE DU T
         DECRET N° 96-203 DU 7 MARS 1996 RELATIF A LA DUREE DU TRAVAIL L'HORAIRE COLLECTIF DE TRAVAIL, Art. 2 — ARTICLE
  0.0323 | Art. 3 | DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT 
         DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A LA DUREE DE LA PERIODE D'ESSAI, Art. 3 —
  0.0320 | Art. 5 | DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT 
         DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A LA DUREE DE LA PERI

### Chuncking final

In [1]:
"""
PHASE 2 — CHUNKING v4 : RÉFÉRENCES HONNÊTES
Pourquoi (découvertes de l'inondation "article 2") :
  1. La section "loi" contient des LOIS ANNEXÉES après le code (travail des
     enfants, etc.) — leurs articles étaient étiquetés "Code du travail" :
     citation MENSONGÈRE, inacceptable pour un outil juridique.
  2. La section "convention" contient des textes récents annexés (ex : décret
     télétravail 2022) — bien détectés comme parents en v3, mais l'
     enrichissement les écrasait avec "Convention collective".
Changements v4 :
  a. RE_TEXTE_PARENT reconnaît aussi "LOI N° ..." → les lois annexées
     deviennent des parents ; le code lui-même a pour parent son propre titre
     ("LOI N° 2015-532..."), normalisé en référence courte "Code du travail".
  b. Nouveau champ `reference` par chunk : la référence de citation HONNÊTE
     (parent détecté si présent, sinon libellé de la section). C'est ce champ
     que l'enrichissement et les citations utiliseront désormais.
Sortie : chunks.json v4 → nécessite ré-indexation (cellule suivante).
"""

from pathlib import Path
import re
import json

RACINE = Path(r"C:\Users\KONAN GERVAIS\Desktop\juriprudence\corpus")
EXTRAITS = RACINE / "extraits"
SORTIE = RACINE / "chunks.json"

RE_ARTICLE = re.compile(
    r"^(?:Art\.|Article|ARTICLE)\s+(\d+(?:\.\d+)?|PREMIER|premier)\s*",
    re.MULTILINE,
)

# v4 : LOI ajoutée aux types de textes parents ET à la liste STOP
STOP = r"(?!ARTICLE|Article|Art\.|DECRET|ARRETE|ORDONNANCE|LOI|ANNEXE|TITRE|CHAPITRE|SECTION)"
RE_TEXTE_PARENT = re.compile(
    r"^((?:DECRET|ARRETE|ORDONNANCE|LOI)\s+N°\s*[^\n]+"
    rf"(?:\n{STOP}[A-ZÉÈÀÇÔÎ0-9][^\n]*)*)",
    re.MULTILINE,
)

STATUTS = {
    "loi": "Loi n°2015-532 du 20/07/2015 (Code du travail)",
    "decrets": "Décret d'application",
    "agence_emploi_jeunes": "Textes Agence Emploi Jeunes",
    "convention": "Convention collective interprofessionnelle du 19/07/1977",
}
# Référence courte par défaut quand aucun parent n'est détecté
REF_DEFAUT = {
    "loi": "Code du travail",
    "decrets": "Décret d'application (non identifié)",
    "agence_emploi_jeunes": "Textes Agence Emploi Jeunes",
    "convention": "Convention collective interprofessionnelle",
}


def reference_de(nom, parent):
    """Détermine la référence de citation honnête d'un chunk.

    Règles :
      - parent contenant "2015-532" → c'est le code lui-même → "Code du
        travail" (référence courte, plus lisible que le titre complet) ;
      - autre parent détecté → son titre complet (loi annexée, décret,
        arrêté, ordonnance — chacun cité sous SON nom) ;
      - aucun parent → libellé par défaut de la section.

    Args:
        nom: section du chunk.
        parent: titre du texte parent détecté, ou None.

    Returns:
        str: la référence à utiliser dans l'enrichissement et les citations.
    """
    if parent and "2015-532" in parent:
        return "Code du travail"
    if parent:
        return parent
    return REF_DEFAUT[nom]


def chunker_section(nom, texte):
    """Découpe une section en chunks (un article chacun), parents rattachés.

    Args:
        nom: clé de la section.
        texte: contenu du fichier .txt.

    Returns:
        Liste de dicts : id, section, statut, texte_parent, reference,
        article, contenu, nb_caracteres.
    """
    chunks = []
    parents = [
        (m.start(), re.sub(r"\s*\n\s*", " ", m.group(1)).strip())
        for m in RE_TEXTE_PARENT.finditer(texte)
    ]

    def parent_de(pos):
        """Titre du dernier texte parent commençant avant `pos` (ou None)."""
        courant = None
        for debut, titre in parents:
            if debut <= pos:
                courant = titre
            else:
                break
        return courant

    matches = list(RE_ARTICLE.finditer(texte))
    for i, m in enumerate(matches):
        debut = m.start()
        fin = matches[i + 1].start() if i + 1 < len(matches) else len(texte)
        contenu = texte[debut:fin].strip()
        numero = m.group(1)
        if numero.upper() == "PREMIER":
            numero = "1"
        parent = parent_de(debut)
        chunks.append({
            "id": f"{nom}_{i:04d}",
            "section": nom,
            "statut": STATUTS[nom],
            "texte_parent": parent,
            "reference": reference_de(nom, parent),
            "article": numero,
            "contenu": contenu,
            "nb_caracteres": len(contenu),
        })
    return chunks


tous = []
for nom in STATUTS:
    texte = (EXTRAITS / f"{nom}.txt").read_text(encoding="utf-8")
    chunks = chunker_section(nom, texte)
    tous.extend(chunks)
    refs = {c["reference"] for c in chunks}
    print(f"{nom:22s} : {len(chunks):4d} chunks | {len(refs)} références distinctes")

SORTIE.write_text(json.dumps(tous, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n→ {len(tous)} chunks (v4) écrits")

# Contrôle immédiat : les références de la section "loi"
print("\nRéférences distinctes dans la section loi :")
for r in sorted({c["reference"] for c in tous if c["section"] == "loi"}):
    print(f"  • {r[:100]}")

loi                    :  406 chunks | 3 références distinctes
decrets                :  528 chunks | 36 références distinctes
agence_emploi_jeunes   :   42 chunks | 2 références distinctes
convention             :  117 chunks | 2 références distinctes

→ 1093 chunks (v4) écrits

Références distinctes dans la section loi :
  • Code du travail
  • LOI N° 2010-272 DU 30 SEPTEMBRE 2010 PORTANT INTERDICTION DE LA TRAITE ET DES PIRES FORMES DE TRAVAI
  • LOI N° 96-670 DU 29 AOUT 1996 PORTANT SUSPENSION DES DELAIS DE SAISINE, DE PRESCRIPTION, DE PEREMPTI


In [2]:
"""
PHASE 4a-v2 — RÉ-INDEXATION SUR LES RÉFÉRENCES HONNÊTES

Pourquoi cette étape : chunks.json v4 porte désormais un champ `reference`
fiable — les lois annexées identifiées (loi 2010-272 sur le travail des
enfants, loi 96-670 sur les délais), le décret télétravail 2022 cité sous
son vrai nom. Mais les index (dense + BM25) ont été construits sur les
anciens passages enrichis, avec les références écrasées par section.
On reconstruit donc les DEUX index avec l'enrichissement basé sur le champ
`reference`, et on stocke `reference` dans les métadonnées ChromaDB pour
les citations de la Phase 5.

Après cette cellule : relancer les cellules des fonctions recherche_hybride
(4b) et recherche/routeur (4d) — leur code est inchangé, mais elles doivent
capturer les nouveaux objets `collection`, `bm25`, `ids_ordre`, `tokeniser`.

Contrôle de succès : "Index reconstruits : 1093 passages (références v4)".
"""

import json
import re
from pathlib import Path

import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

RACINE = Path(r"C:\Users\KONAN GERVAIS\Desktop\juriprudence\corpus")
CHUNKS = RACINE / "chunks.json"
INDEX_DIR = str(RACINE / "index_chroma")
MODELE = "intfloat/multilingual-e5-base"


def texte_enrichi(c):
    """Construit le passage enrichi : référence honnête + contenu.

    v2 : utilise le champ `reference` calculé au chunking v4 (texte parent
    détecté, ou libellé par défaut de la section) — fini les références
    écrasées par section. Chaque article s'annonce sous le nom du texte
    auquel il appartient réellement.

    Args:
        c: un chunk (dict) de chunks.json v4.

    Returns:
        str: "référence, Art. X — contenu", prêt à encoder/indexer.
    """
    return f"{c['reference']}, Art. {c['article']} — {c['contenu']}"


def tokeniser(texte):
    """Tokenise pour BM25 en préservant les nombres pointés.

    "14.5" reste UN token (rare → fort IDF → discriminant pour les
    références exactes), au lieu d'être éclaté en "14" + "5" (banals).

    Args:
        texte: chaîne à tokeniser (passage ou question).

    Returns:
        Liste de tokens en minuscules.
    """
    return re.findall(r"\d+\.\d+|\w+", texte.lower())


# --- 1. Charger chunks v4 et modèle ---
chunks = json.loads(CHUNKS.read_text(encoding="utf-8"))
modele = SentenceTransformer(MODELE)
passages = [texte_enrichi(c) for c in chunks]
print(f"{len(passages)} passages enrichis (références v4) — exemple :")
print(passages[60][:150], "\n")

# --- 2. Encoder (dense) et reconstruire ChromaDB ---
embeddings = modele.encode(
    [f"passage: {p}" for p in passages],
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

client = chromadb.PersistentClient(path=INDEX_DIR)
try:
    client.delete_collection("droit_ivoirien")
except Exception:
    pass
collection = client.create_collection(
    name="droit_ivoirien",
    metadata={"hnsw:space": "cosine"},
)
collection.add(
    ids=[c["id"] for c in chunks],
    embeddings=embeddings.tolist(),
    documents=passages,
    metadatas=[{
        "section": c["section"],
        "statut": c["statut"],
        "texte_parent": c["texte_parent"] or "",
        "reference": c["reference"],
        "article": c["article"],
    } for c in chunks],
)

# --- 3. Index BM25 sur les mêmes passages ---
bm25 = BM25Okapi([tokeniser(p) for p in passages])
ids_ordre = [c["id"] for c in chunks]

print(f"Index reconstruits : {collection.count()} passages (références v4)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

1093 passages enrichis (références v4) — exemple :
Code du travail, Art. 15.1 — Art. 15.1
Le contrat de travail à durée déterminée est un contrat qui prend fin à l'arrivée d'un
terme fixé par les parti 



Batches:   0%|          | 0/35 [00:00<?, ?it/s]

Index reconstruits : 1093 passages (références v4)


In [20]:
"""
CONTRÔLE FINAL PHASE 4 — les références honnêtes en conditions réelles.
Vérifie : (1) le trio période d'essai toujours en tête (pas de régression),
(2) les articles des lois annexées cités sous leur VRAI nom,
(3) le décret télétravail 2022 correctement identifié.
"""

questions = [
    "Que prévoit l'article 2 sur la période d'essai ?",
    "Le travail des enfants est-il interdit ?",           # loi 2010-272 attendue
    "Le télétravail est-il encadré ?",                    # décret 2022-31 attendu
]
for q in questions:
    print(f"\n{'='*70}\nQ : {q}")
    for id_, s, doc, meta in recherche(q, k=5):
        marque = "🎯" if s == 1.0 else "  "
        print(f"{marque}{s:.4f} | Art. {meta['article']} | {meta['reference'][:60]}")
        print(f"         {doc[:110]}")


Q : Que prévoit l'article 2 sur la période d'essai ?
🎯1.0000 | Art. 2 | DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'E
         DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A LA DUREE DE LA PERIODE D'ESSAI, Art. 2 —
🎯1.0000 | Art. 2 | DECRET N° 96-203 DU 7 MARS 1996 RELATIF A LA DUREE DU TRAVAI
         DECRET N° 96-203 DU 7 MARS 1996 RELATIF A LA DUREE DU TRAVAIL L'HORAIRE COLLECTIF DE TRAVAIL, Art. 2 — ARTICLE
🎯1.0000 | Art. 2 | DECRET N° 96-206 DU 7 MARS 1996 RELATIF AU COMITE D'HYGIENE,
         DECRET N° 96-206 DU 7 MARS 1996 RELATIF AU COMITE D'HYGIENE, DE SECURITE ET DES CONDITIONS DE TRAVAIL, Art. 2 
  0.0323 | Art. 3 | DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'E
         DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A LA DUREE DE LA PERIODE D'ESSAI, Art. 3 —
  0.0320 | Art. 5 | DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'E
         DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT 

# Etape de Génération


PHASE 5 (version locale) — GÉNÉRATION VIA OLLAMA

Pourquoi cette étape (règles inchangées par rapport à l'objectif) :
  1. FIDÉLITÉ : répondre UNIQUEMENT à partir des passages fournis.
  2. CITATIONS : chaque affirmation adossée à sa source.
  3. ABSTENTION : si les passages ne répondent pas, le dire (leçon du test TVA).
Dans un premier temps, nous avons choisi ollama 3.1 8B.
Choix technique : LLM local via Ollama (Llama 3.1 8B) — gratuit, illimité,
confidentiel par construction (les données ne quittent pas la machine — argument
fort pour un outil juridique). Contrepartie : un 8B suit moins strictement les
règles de citation/abstention qu'un grand modèle d'API ; le prompt doit être
d'autant plus directif, et la Phase 6 mesurera objectivement la fidélité.

Prérequis :
  - Ollama installé et lancé (service sur localhost:11434) ;
  - Modèle téléchargé : `ollama pull llama3.2:3b` ;
  - Client universel : `pip install openai`.

Note perf : Les réponses étaient très lentes. j'ai par suite pris on modèle plus leger (ollama 3.2:3B) c'était parait avec des rédactions moins convaincantes. J'ai pour finir pris un modèle existant sur Hagging Face :llama-3.3-70b-versatile. Très rapide gratuit avec une meilleure qualité de redaction.



In [21]:
pip install OpenAI -q

Note: you may need to restart the kernel to use updated packages.


In [22]:
pip install python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


# réponses obtenues avec llama3:1 8B

J'ai testé llama3.1 8b dans un premier temps, Cela a fonctionné mais beaucoup lent(25 mins pour seulement 3 questions). Voici la réponse générée:

Q : Quelle est la durée maximale de la période d'essai ?

La durée maximale de la période d'essai varie selon la catégorie professionnelle à laquelle appartient le travailleur, comme indiqué dans l'article 14 de la Convention collective interprofessionnelle [2] :

* Ouvriers et employés : 8 jours pour les travailleurs payés à l'heure ; 1 mois pour les travailleurs payés au mois ;
* Agents de maîtrise, techniciens et assimilés : 2 mois ;
* Ingénieurs, Cadres et assimilés : 3 mois ;
* Cadres supérieurs : 6 mois.

⚠️ Information documentaire — ne constitue pas un conseil juridique.
Consultez un professionnel du droit pour votre situation.

--- 8 passages consultés ---
[1]    Code du travail, Art. 14.5
[2]    Convention collective interprofessionnelle, Art. 14
[3]    DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A , Art. 3
[4]    DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A , Art. 7
[5]    DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A , Art. 4
[6]    DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A , Art. 5
[7]    DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A , Art. 6
[8]    Convention collective interprofessionnelle, Art. 15

======================================================================

Q : Mon patron peut-il me virer sans préavis ?

Les textes dont je dispose ne permettent pas de répondre à cette question.

--- 8 passages consultés ---
[1]    Code du travail, Art. 18.7
[2]    Code du travail, Art. 82.5
[3]    Convention collective interprofessionnelle, Art. 35
[4]    Code du travail, Art. 51.8
[5]    Convention collective interprofessionnelle, Art. 32
[6]    Code du travail, Art. 13.10
[7]    Code du travail, Art. 32.7
[8]    Convention collective interprofessionnelle, Art. 27

======================================================================

Q : Quel est le taux de TVA en Côte d'Ivoire ?

Les textes dont je dispose ne me permettent pas de répondre à cette question.

--- 8 passages consultés ---
[1]    LOI N° 2010-272 DU 30 SEPTEMBRE 2010 PORTANT INTERDICTION DE LA TRAITE, Art. 29
[2]    Convention collective interprofessionnelle, Art. 57
[3]    LOI N° 2010-272 DU 30 SEPTEMBRE 2010 PORTANT INTERDICTION DE LA TRAITE, Art. 28
[4]    Convention collective interprofessionnelle, Art. 58
[5]    DECRET N° 96-287 DU 3 AVRIL 1996 RELATIF AU CONTRAT DE TRAVAIL, Art. 14
[6]    Code du travail, Art. 1
[7]    DECRET N° 2014-370 DU 18 JUIN 2014 RELATIF AU REGIME DE LA QUOTITE CES, Art. 4
[8]    DECRET N°

## Réponse de llama 3.2:3B

Q : Quelle est la durée maximale de la période d'essai ?

[1] Code du travail, Art. 14.5 — Art. 14.5

La durée maximale de la période d'essai varie selon la catégorie professionnelle à laquelle appartient le travailleur.

Pour les ouvriers et employés payés à l'heure : 8 jours
Pour les ouvriers et employés payés au mois : 1 (un) mois
Pour les agents de maîtrise, techniciens et assimilés : 2 mois
Pour les ingénieurs, cadres et assimilés : 3 mois
Pour les cadres supérieurs : 6 mois.

--- 8 passages consultés ---
[1]    Code du travail, Art. 14.5
[2]    Convention collective interprofessionnelle, Art. 14
[3]    DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A , Art. 3
[4]    DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A , Art. 7
[5]    DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A , Art. 4
[6]    DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A , Art. 5
[7]    DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A , Art. 6
[8]    Convention collective interprofessionnelle, Art. 15

======================================================================

Q : Mon patron peut-il me virer sans préavis ?

⚠️ Information documentaire — ne constitue pas un conseil juridique.
Consultez un professionnel du droit pour votre situation.

Selon l'article 18.7 du Code du travail, une rupture de contrat à durée indéterminée sans préavis ou sans que le délai de préavis ait été intégralement observé emporte obligation pour la partie responsable de verser à l'autre une indemnité correspondant à la rémunération et aux avantages de toute nature dont aurait bénéficié le travailleur durant le délai de préavis non respecté.

--- 8 passages consultés ---
[1]    Code du travail, Art. 18.7
[2]    Code du travail, Art. 82.5
[3]    Convention collective interprofessionnelle, Art. 35
[4]    Code du travail, Art. 51.8
[5]    Convention collective interprofessionnelle, Art. 32
[6]    Code du travail, Art. 13.10
[7]    Code du travail, Art. 32.7
[8]    Convention collective interprofessionnelle, Art. 27

======================================================================

Q : Quel est le taux de TVA en Côte d'Ivoire ?

⚠️ Information documentaire — ne constitue pas un conseil juridique.
Consultez un professionnel du droit pour votre situation.

[6] Code du travail, Art. 1 — Article 1 n'est pas pertinent pour la question de taux de TVA mais [8] DECRET N° 2022-31 DU 12 JANVIER 2022 FIXANT LES MODALITES DE MISE EN ŒUVRE DU TELETRAVAIL POUR LES TRAVAILLEURS REGIS PAR LE CODE DU TRAVAIL, Art. 11 — ARTICLE 11 n'est pas pertinent non plus.

Cependant [8] mentionne le Ministre de l'Emploi et de la Protection Sociale et le Ministre de l'Economie Numérique, des Télécommunications et de l'innovation chargés de l'exécution du décret.

[7] DECRET N° 2014-370 DU 18 JUIN 2014 RELATIF AU REGIME DE LA QUOTITE CESSIBLE ET DE LA QUOTITE SAISISSABLE, Art. 4 — ARTICLE 4 mentionne les taux de TVA mais pas le taux applicable en Côte d'Ivoire.

Cependant [7] mentionne que les taux de TVA sont fixés à 35% pour les salaires inférieurs ou égaux à 200.000 FCFA, et augmentent progressivement jusqu'à 57% pour les salaires supérieurs à 2 000 000 FCFA.

Il n'est pas possible d'établir avec précision le taux de TVA applicable en Côte d'Ivoire sans une source plus spécifique.

Les textes dont je dispose ne me permettent pas de répondre à cette question.

--- 8 passages consultés ---
[1]    LOI N° 2010-272 DU 30 SEPTEMBRE 2010 PORTANT INTERDICTION DE LA TRAITE, Art. 29
[2]    Convention collective interprofessionnelle, Art. 57
[3]    LOI N° 2010-272 DU 30 SEPTEMBRE 2010 PORTANT INTERDICTION DE LA TRAITE, Art. 28
[4]    Convention collective interprofessionnelle, Art. 58
[5]    DECRET N° 96-287 DU 3 AVRIL 1996 RELATIF AU CONTRAT DE TRAVAIL, Art. 14
[6]    Code du travail, Art. 1
[7]    DECRET N° 2014-370 DU 18 JUIN 2014 RELATIF AU REGIME DE LA QUOTITE CES, Art. 4
[8]    DECRET N° 2022-31 DU 12 JANVIER 2022 FIXANT LES MODALITES DE MISE EN Œ, Art. 11

## Conclusion sur ce test

Question 1 — période d'essai
llama 3.2:3B : réponse structurée par catégorie, citation implicite (« Art. 14.5 » en tête du passage), avertissement à la fin. llama3.1:8B : mêmes durées, mieux introduit (« varie selon la catégorie... »), citation explicite [2], avertissement à la fin. Les deux réussissent, le 8B rédige un peu mieux.
Question 2 — virer sans préavis
3B : cite correctement l'Art. 18.7, paraphrase fidèlement (indemnité compensatrice). Réponse juste et sourcée. ✓
8B : s'abstient à tort — la réponse est en position [1] et il ne la voit pas. Faux négatif d'abstention.
Question 3 — TVA
3B : hallucine — tente d'analyser passage par passage, invente que l'article 4 du décret 2014-370 « mentionne les taux de TVA fixés à 35%... 57% » (faux, cet article parle de quotités saisissables), puis termine par la phrase d'abstention. Grave : chiffre juridique inventé.
8B : abstient proprement, phrase exacte. ✓
Le vrai verdict, cette fois
Ni le 3B ni le 8B n'est fiable — mais l'analyse tient toujours : chacun rate à sa façon la règle la plus critique. Le 3B est meilleur sur la fidélité aux passages (question 2) mais hallucine sur les questions ambiguës (question 3). Le 8B est plus strict sur l'abstention (question 3) mais s'abstient à tort quand la réponse est là (question 2). Aucun des deux n'a le comportement attendu : répondre quand la réponse est présente, s'abstenir quand elle ne l'est pas. C'est précisément la métrique que la Phase 6 mesurera.
Ce petit test à 3 questions ne suffit évidemment pas à conclure — le 3B pourrait avoir été chanceux sur Q2 et le 8B sur Q3. C'est justement pourquoi on construit un jeu d'évaluation plus large : sur 30 questions bien choisies, la vraie tendance de chaque modèle apparaîtra.

## Vers d'autres modèle rapide, précis.

In [53]:

"""
PHASE 5 (version locale) — GÉNÉRATION VIA OLLAMA
Pourquoi cette étape (règles inchangées par rapport à l'objectif) :
  1. FIDÉLITÉ : répondre UNIQUEMENT à partir des passages fournis.
  2. CITATIONS : chaque affirmation adossée à sa source.
  3. ABSTENTION : si les passages ne répondent pas, le dire (leçon du test TVA).

Choix technique : LLM local via Ollama (Llama 3.1 8B) — gratuit, illimité,
confidentiel par construction (les données ne quittent pas la machine — argument
fort pour un outil juridique). Contrepartie : un 8B suit moins strictement les
règles de citation/abstention qu'un grand modèle d'API ; le prompt doit être
d'autant plus directif, et la Phase 6 mesurera objectivement la fidélité.

Prérequis :
  - Ollama installé et lancé (service sur localhost:11434) ;
  - Modèle téléchargé : `llama-3.3-70b-versatile` ;
  - Client universel : `pip install openai`.


"""

from openai import OpenAI
import os

# Client OpenAI branché sur l'API locale d'Ollama.
# api_key est requis par le client mais ignoré par Ollama : valeur factice.

from dotenv import load_dotenv
load_dotenv()

client_llm = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)
MODELE_LLM = "llama-3.3-70b-versatile"

SYSTEM = """Tu es un assistant documentaire spécialisé en droit du travail ivoirien.

RÈGLES IMPÉRATIVES (à respecter scrupuleusement) :
1. Tu réponds UNIQUEMENT à partir des extraits fournis ci-dessous. Tu n'utilises
   JAMAIS tes connaissances générales, même si tu penses connaître la réponse.
   Un droit ivoirien inventé ou emprunté au droit français est une faute grave.
2. Chaque affirmation cite sa source entre crochets, en reprenant les références
   exactes des extraits : [Code du travail, Art. 14.5] ou
   [Décret n°96-195 du 7 mars 1996, Art. 2].
3. Si les extraits ne permettent pas de répondre à la question posée, tu réponds
   EXACTEMENT cette phrase, sans en ajouter d'autres :
   "Les textes dont je dispose ne me permettent pas de répondre à cette question."
4. Si plusieurs textes se complètent (loi + décret + convention), tu les
   présentes dans cet ordre hiérarchique.
5. Tu termines TOUJOURS par cet avertissement, mot pour mot :
   "⚠️ Information documentaire — ne constitue pas un conseil juridique.
   Consultez un professionnel du droit pour votre situation."

Réponds de façon concise (5 à 15 lignes maximum, sauf question à réponse
hiérarchisée)."""


def repondre(question, k=8, afficher_sources=True):
    """Chaîne RAG complète : retrieval hybride+routeur → génération sourcée.

    Args:
        question: question en langage naturel.
        k: nombre de passages transmis au LLM (8 = assez large pour couvrir
           loi + décrets + convention sur un même sujet).
        afficher_sources: si True, liste les passages utilisés après la
           réponse (transparence + débogage).

    Returns:
        str: la réponse rédigée du LLM.
    """
    # Retrieval : la fonction `recherche` définie en Phase 4 (routeur + hybride)
    passages = recherche(question, k=k)

    # 2. Mise en forme du contexte : passages numérotés [1..k]
    contexte = "\n\n".join(
        f"[{i+1}] {doc}" for i, (_, _, doc, _) in enumerate(passages)
    )
    message = (
        f"EXTRAITS DES TEXTES JURIDIQUES IVOIRIENS :\n\n{contexte}\n\n"
        f"QUESTION : {question}"
    )

    # 3. Appel du LLM local (API OpenAI-compatible)
    reponse = client_llm.chat.completions.create(
        model=MODELE_LLM,
        temperature=0.1,   # quasi déterministe : fidélité > créativité
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": message},
        ],
    )
    texte = reponse.choices[0].message.content

    # 4. Affichage
    print(f"Q : {question}\n")
    print(texte)
    if afficher_sources:
        print(f"\n--- {len(passages)} passages consultés ---")
        for i, (_, s, _, meta) in enumerate(passages):
            marque = "🎯" if s == 1.0 else "  "
            print(f"[{i+1}] {marque} {meta['reference'][:70]}, Art. {meta['article']}")
    return texte


# --- Le baptême du feu : trois profils de questions ---
# Test 1 : réponse hiérarchisée attendue (loi → décret → convention)
_ = repondre("Quelle est la durée maximale de la période d'essai ?")
print("\n" + "=" * 70 + "\n")

# Test 2 : reformulation argotique + retrieval partiellement bruité
_ = repondre("Mon patron peut-il me virer sans préavis ?")
print("\n" + "=" * 70 + "\n")

# Test 3 : abstention obligatoire (hors corpus — le vrai test pour évaluer notre
# modèle)
_ = repondre("Quel est le taux de TVA en Côte d'Ivoire ?")

Q : Quelle est la durée maximale de la période d'essai ?

La durée maximale de la période d'essai varie selon la catégorie professionnelle [Convention collective interprofessionnelle, Art. 14]. Elle est fixée comme suit :
- 8 jours pour les travailleurs payés à l'heure (ouvriers et employés) [Convention collective interprofessionnelle, Art. 14] ;
- 1 mois pour les travailleurs payés au mois (ouvriers et employés) [Convention collective interprofessionnelle, Art. 14] ;
- 2 mois pour les agents de maîtrise, techniciens et assimilés [Convention collective interprofessionnelle, Art. 14] ;
- 3 mois pour les ingénieurs, cadres et assimilés [Convention collective interprofessionnelle, Art. 14] ;
- 6 mois pour les cadres supérieurs [Convention collective interprofessionnelle, Art. 14].
Ces délais ne sont renouvelables qu'une seule fois [Convention collective interprofessionnelle, Art. 14].
⚠️ Information documentaire — ne constitue pas un conseil juridique. Consultez un professionnel du droit 

Analyse des trois réponses
Question 1 — période d'essai. Réponse structurée à la perfection : introduction, énumération claire des cinq catégories, chaque ligne citée, information bonus sur le renouvellement (« ne sont renouvelables qu'une seule fois »), avertissement à la fin. Chaque affirmation est adossée à sa source. C'est de la qualité produit.
Question 2 — virer sans préavis. Cite l'Art. 18.7 sur la conséquence financière (indemnité compensatrice), puis ajoute la nuance clé : « la rupture peut intervenir sans préavis en cas de faute lourde ». C'est exactement la réponse qu'un juriste donnerait. Là où le 8B s'abstenait à tort et le 3B répondait mais sans la nuance faute lourde, le 70B fait les deux : réponse complète et nuancée.
Question 3 — TVA. Phrase d'abstention exacte, aucune tergiversation, aucune hallucination. Le contraste avec le 8B qui inventait des taux de TVA depuis un article sur les quotités saisissables est saisissant.
Score : 3 sur 3. Fidélité, citations, abstention — les trois exigences du cahier des charges tenues sur les trois profils de questions. Et en 2 secondes chrono par réponse.

# Phase 6: évaluation
Maintenant qu'on a un système rapide et fiable, il est temps de le mesurer objectivement — pas juste « ça marche sur mes 3 questions préférées », mais des scores chiffrés sur un jeu de questions représentatif.

Trois métriques à mesurer :

Recall@k du retrieval — sur des questions dont on connaît l'article attendu, est-il dans les k passages remontés ?
Fidélité des réponses — chaque affirmation est-elle vraiment soutenue par les passages cités ? (Vérification manuelle sur un petit échantillon.)
Justesse de l'abstention — sur les hors-corpus, s'abstient-il ? Et inversement, ne s'abstient-il pas à tort ?

In [36]:
"""
PHASE 6a — CONSTITUTION DU JEU D'ÉVALUATION
============================================
Pourquoi : sans jeu de questions annoté, aucune métrique n'a de sens. Chaque
entrée fixe la réponse attendue AVANT toute mesure — c'est le contrat qui
protège contre l'auto-tromperie.

Trois catégories, chacune sondant une hypothèse précise :
  - COUVERT : questions dont la réponse existe clairement dans le corpus.
    Métrique clé : recall@k. Score attendu : élevé (>90% sur un bon système).
  - DIFFICILE : reformulations argotiques, références exactes, questions
    ambiguës. Métrique clé : recall@k sur les cas où les phases 3-4 avaient
    identifié des faiblesses. Le tri des faux échecs = boussole des futures
    améliorations.
  - HORS-CORPUS : questions sans réponse dans le corpus. Métrique clé :
    justesse de l'abstention. Score attendu : le LLM produit la phrase
    d'abstention exacte, sans hallucinations.
"""

import json
from pathlib import Path

RACINE = Path(r"C:\Users\KONAN GERVAIS\Desktop\juriprudence")
EVAL_DIR = RACINE / "eval"
EVAL_DIR.mkdir(exist_ok=True)

QUESTIONS = [
    # COUVERT (5)
    {
        "id": "cov_01",
        "categorie": "couvert",
        "question": "Quelle est la durée maximale de la période d'essai ?",
        "article_attendu": {"reference": "DECRET N° 96-195", "article": "2"},
        "articles_pertinents": [
            {"reference": "Code du travail", "article": "14.5"},
            {"reference": "Convention", "article": "14"},
        ],
        "commentaire": "Réponse distribuée loi + décret + convention — teste la hiérarchie.",
    },
    {
        "id": "cov_02",
        "categorie": "couvert",
        "question": "Une femme enceinte peut-elle être licenciée ?",
        "article_attendu": {"reference": "Code du travail", "article": "23.4"},
        "articles_pertinents": [
            {"reference": "Code du travail", "article": "23.6"},
            {"reference": "Code du travail", "article": "23.8"},
        ],
        "commentaire": "Reformulation (licencier ≠ résilier) — test sémantique classique.",
    },
    {
        "id": "cov_03",
        "categorie": "couvert",
        "question": "Combien de jours de congés payés par mois de travail ?",
        "article_attendu": {"reference": "Code du travail", "article": "25.1"},
        "articles_pertinents": [
            {"reference": "Convention", "article": "69"},
        ],
        "commentaire": "Question factuelle avec chiffre précis attendu (2,2 jours).",
    },
    {
        "id": "cov_04",
        "categorie": "couvert",
        "question": "Le travail des enfants est-il interdit ?",
        "article_attendu": {"reference": "LOI N° 2010-272", "article": "7"},
        "articles_pertinents": [
            {"reference": "LOI N° 2010-272", "article": "5"},
            {"reference": "ARRETE N° 009", "article": "5"},
        ],
        "commentaire": "Test des LOIS ANNEXÉES — vérifie que la loi 2010-272 est bien citée sous son nom (pas Code du travail).",
    },
    {
        "id": "cov_05",
        "categorie": "couvert",
        "question": "Comment se calcule l'indemnité de licenciement ?",
        "article_attendu": {"reference": "DECRET N° 2017-210", "article": "3"},
        "articles_pertinents": [{"reference": "DECRET N° 96-201", "article": "3"},
        {"reference": "Convention collective interprofessionnelle", "article": "39"},
        {"reference": "Code du travail", "article": "18.11"},
        ],
        "commentaire": "Réponse distribuée : le calcul est dans l'art. 3 du décret 2017-210 (le nouveau). L'art. 3 du décret 96-201 (l'ancien, abrogé) et l'art. 39 de la convention traitent aussi du barème."
    },

    # DIFFICILE (5)
    {
        "id": "dif_01",
        "categorie": "difficile",
        "question": "Que dit l'article 14.5 du Code du travail ?",
        "article_attendu": {"reference": "Code du travail", "article": "14.5"},
        "articles_pertinents": [],
        "commentaire": "RÉFÉRENCE EXACTE — teste le routeur avec filtre section.",
    },
    {
        "id": "dif_02",
        "categorie": "difficile",
        "question": "Mon patron peut-il me virer sans préavis ?",
        "article_attendu": {"reference": "Code du travail", "article": "18.7"},
        "articles_pertinents": [
            {"reference": "Code du travail", "article": "16.11"},  # faute lourde
        ],
        "commentaire": "REGISTRE ARGOTIQUE (virer) — teste la robustesse au registre.",
    },
    {
        "id": "dif_03",
        "categorie": "difficile",
        "question": "Est-ce que le télétravail est encadré en Côte d'Ivoire ?",
        "article_attendu": {"reference": "DECRET N° 2022-31", "article": "2"},
        "articles_pertinents": [
            {"reference": "DECRET N° 2022-31", "article": "3"},
        ],
        "commentaire": "Test de la référence RÉCENTE (2022) mal étiquetée au départ.",
    },
    {
        "id": "dif_04",
        "categorie": "difficile",
        "question": "Que prévoit l'article 2 de la convention collective ?",
        "article_attendu": {"reference": "Convention collective interprofessionnelle", "article": "2"},
        "articles_pertinents": [],
        "commentaire": "RÉFÉRENCE AMBIGUË (44 art.2 dans le corpus) — teste filtre section + désambiguïsation.",
    },
    {
        "id": "dif_05",
        "categorie": "difficile",
        "question": "Un salarié peut-il faire grève sans prévenir ?",
        "article_attendu": {"reference": "Code du travail", "article": "82.5"},
        "articles_pertinents": [
            {"reference": "Code du travail", "article": "82.6"},
        ],
        "commentaire": "Reformulation (préavis de grève) — sujet sensible, retrieval doit être précis.",
    },

    #HORS-CORPUS (5)
    {
        "id": "hc_01",
        "categorie": "hors_corpus",
        "question": "Quel est le taux de TVA en Côte d'Ivoire ?",
        "attend_abstention": True,
        "commentaire": "Fiscalité — hors du droit du travail. Test d'abstention canonique.",
    },
    {
        "id": "hc_02",
        "categorie": "hors_corpus",
        "question": "Quelle est la peine pour vol simple en Côte d'Ivoire ?",
        "attend_abstention": True,
        "commentaire": "Droit pénal — proche thématiquement (peines mentionnées dans le code) mais hors sujet.",
    },
    {
        "id": "hc_03",
        "categorie": "hors_corpus",
        "question": "Qui est le président actuel de la Côte d'Ivoire ?",
        "attend_abstention": True,
        "commentaire": "Actualité politique — hors nature du corpus.",
    },
    {
        "id": "hc_04",
        "categorie": "hors_corpus",
        "question": "Quel est le prix du timbre fiscal pour un passeport ?",
        "attend_abstention": True,
        "commentaire": "Administratif — piégeux car 'fiscal' peut attirer des passages sur les indemnités.",
    },
    {
        "id": "hc_05",
        "categorie": "hors_corpus",
        "question": "Comment créer une SARL en Côte d'Ivoire ?",
        "attend_abstention": True,
        "commentaire": "Droit des sociétés (OHADA) — proche du travail mais nettement distinct.",
    },
]

fichier = EVAL_DIR / "questions.json"
fichier.write_text(json.dumps(QUESTIONS, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{len(QUESTIONS)} questions écrites dans {fichier}")
print(f"\nRépartition :")
from collections import Counter
for cat, n in Counter(q["categorie"] for q in QUESTIONS).items():
    print(f"  {cat:15s} : {n}")

15 questions écrites dans C:\Users\KONAN GERVAIS\Desktop\juriprudence\eval\questions.json

Répartition :
  couvert         : 5
  difficile       : 5
  hors_corpus     : 5


In [37]:
"""
VÉRIFICATION MANUELLE DES ARTICLES ATTENDUS

Pourquoi : la feuille de réponses ("l'article X est LA réponse à la question Y")
doit être JUSTE avant toute mesure. Un article attendu faux fausserait le
recall pour toujours.

Méthode : pour chaque question COUVERT et DIFFICILE, on affiche le contenu de
l'article annoté.
"""

import json
from pathlib import Path

RACINE = Path(r"C:\Users\KONAN GERVAIS\Desktop\juriprudence")
questions = json.loads((RACINE / "eval" / "questions.json").read_text(encoding="utf-8"))

# On utilise directement ChromaDB (déjà chargé en mémoire depuis la Phase 4).
for q in questions:
    if q["categorie"] == "hors_corpus":
        continue
    art = q["article_attendu"]
    # Recherche par métadonnées : article + section (déduite du préfixe de reference)
    res = collection.get(where={"article": {"$eq": art["article"]}}, limit=50)
    # Filtre sur la référence (substring, plus tolérant qu'égalité stricte)
    trouves = [
        (doc, meta) for doc, meta in zip(res["documents"], res["metadatas"])
        if art["reference"].lower() in meta["reference"].lower()
    ]
    print(f"\n{'='*70}")
    print(f"[{q['id']}] {q['question']}")
    print(f"→ Article attendu : {art['reference']}, Art. {art['article']}")
    if not trouves:
        print("  ⚠️ AUCUN CHUNK TROUVÉ — la référence ou le numéro d'article est faux !")
    else:
        for doc, meta in trouves[:2]:  # max 2 si ambiguïté (art. 2 : plusieurs candidats)
            print(f"  ✓ trouvé : {meta['reference'][:70]}")
            print(f"    {doc[:250]}")


[cov_01] Quelle est la durée maximale de la période d'essai ?
→ Article attendu : DECRET N° 96-195, Art. 2
  ✓ trouvé : DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A 
    DECRET N° 96-195 DU 7 MARS 1996 RELATIF A L'ENGAGEMENT A L'ESSAI ET A LA DUREE DE LA PERIODE D'ESSAI, Art. 2 — ARTICLE 2
Dans l'un ou l'autre cas, la durée de l'essai est fixée par écrit comme suit :
huit jours pour les travailleurs payés à l'heure o

[cov_02] Une femme enceinte peut-elle être licenciée ?
→ Article attendu : Code du travail, Art. 23.4
  ✓ trouvé : Code du travail
    Code du travail, Art. 23.4 — Art. 23.4
Aucun employeur ne peut résilier le contrat de travail d'une salariée lorsqu'elle est
en état de grossesse médicalement constaté et pendant l'intégralité des périodes de
suspension du contrat de travail auxquell

[cov_03] Combien de jours de congés payés par mois de travail ?
→ Article attendu : Code du travail, Art. 25.1
  ✓ trouvé : Code du travail
    Code du travail, Art.

## Cap sur les métriques

In [46]:
"""
PHASE 6b — MESURE DU RECALL@k

Pourquoi cette étape : première métrique chiffrée du système. Le recall@k
mesure la capacité du RETRIEVAL à faire remonter l'article attendu dans les
k premiers résultats. C'est le plafond : si le bon article n'est pas dans les
k, le LLM ne le devinera pas.

On mesure à plusieurs profondeurs (k=1, 3, 5, 8) pour observer la vitesse de
récupération. Un score élevé à k=1 = système précis ; un score qui grimpe
lentement = système qui trouve mais mal classe (candidat à un re-ranker).

Convention de succès : un passage remonté valide la question s'il matche
  - la RÉFÉRENCE attendue (substring, insensible à la casse),
  - ET le NUMÉRO D'ARTICLE (égalité stricte).
Les articles_pertinents comptent aussi comme succès (réponses distribuées).
"""

import json
from pathlib import Path
from collections import defaultdict

RACINE = Path(r"C:\Users\KONAN GERVAIS\Desktop\juriprudence")
questions = json.loads((RACINE / "eval" / "questions.json").read_text(encoding="utf-8"))

# Ne mesure pas sur hors_corpus : par définition aucun article n'est attendu.
questions_couvertes = [q for q in questions if q["categorie"] != "hors_corpus"]

def article_matche(meta, cible):
    """Un passage matche-t-il la cible (référence + numéro d'article) ?

    Match tolérant sur la référence (substring, casse ignorée) et strict
    sur le numéro. C'est robuste aux variations de libellé sans être laxiste.
    """
    return (cible["reference"].lower() in meta["reference"].lower()
                          and meta["article"] == cible["article"])
def rangs_succes(question, k_max=10):
    """Pour une question, retourne l'ensemble des rangs où un article valide apparaît.

    Un article valide = article_attendu OU l'un des articles_pertinents.
    On retourne la position (0-indexée) de la PREMIÈRE occurrence de chaque
    article valide dans le top k_max — utile pour le recall@k à plusieurs k.
    """
    cibles = [question["article_attendu"]] + question.get("articles_pertinents", [])
    resultats = recherche(question["question"], k=k_max)
    rangs = []
    for rang, (_, _, _, meta) in enumerate(resultats):
        if any(article_matche(meta, c) for c in cibles):
            rangs.append(rang)
    return rangs


# --- Calcul du recall@k pour k = 1, 3, 5, 8 ---
Ks = [1, 3, 5, 8]
scores_par_categorie = defaultdict(lambda: {k: [] for k in Ks})
details = []

for q in questions_couvertes:
    rangs = rangs_succes(q, k_max=max(Ks))
    premier_rang = min(rangs) if rangs else None
    for k in Ks:
        succes = premier_rang is not None and premier_rang < k
        scores_par_categorie[q["categorie"]][k].append(succes)
    details.append((q, premier_rang))

# --- Affichage ---
print(f"{'CATÉGORIE':<12} {'N':<4} " + " ".join(f"R@{k:<3}" for k in Ks))
print("-" * 42)
for cat, scores in scores_par_categorie.items():
    n = len(scores[Ks[0]])
    ligne = f"{cat:<12} {n:<4} "
    for k in Ks:
        pct = 100 * sum(scores[k]) / n
        ligne += f"{pct:>4.0f}% "
    print(ligne)

# Détail par question : rang du premier match ---
print("\nRang du premier match pour chaque question :")
for q, rang in details:
    statut = f"rang {rang}" if rang is not None else "❌ ABSENT du top 10"
    print(f"  [{q['id']}] {statut:<20} — {q['question'][:60]}")

CATÉGORIE    N    R@1   R@3   R@5   R@8  
------------------------------------------
couvert      5      60%   60%   80%  100% 
difficile    5      60%   80%   80%   80% 

Rang du premier match pour chaque question :
  [cov_01] rang 0               — Quelle est la durée maximale de la période d'essai ?
  [cov_02] rang 0               — Une femme enceinte peut-elle être licenciée ?
  [cov_03] rang 5               — Combien de jours de congés payés par mois de travail ?
  [cov_04] rang 0               — Le travail des enfants est-il interdit ?
  [cov_05] rang 3               — Comment se calcule l'indemnité de licenciement ?
  [dif_01] rang 0               — Que dit l'article 14.5 du Code du travail ?
  [dif_02] rang 0               — Mon patron peut-il me virer sans préavis ?
  [dif_03] ❌ ABSENT du top 10   — Est-ce que le télétravail est encadré en Côte d'Ivoire ?
  [dif_04] rang 0               — Que prévoit l'article 2 de la convention collective ?
  [dif_05] rang 1               — U

Ce que disent les scores
Le bon côté — 6 questions sur 10 atteignent le rang 0 (premier résultat) : période d'essai, femme enceinte, travail des enfants, article 14.5, virer sans préavis, article 2 convention. Y compris les deux victoires signées de la Phase 4 (référence exacte et argot). Le retrieval fonctionne bien sur son cœur de cible.
Le côté à instruire — trois défaillances distinctes :

dif_05 (grève) à rang 1 : trouvé mais pas premier — mineur, l'Art. 82.6 (« Le préavis émane... ») a probablement dominé le 82.5 sur les mots « préavis de grève ».
cov_03 (2,2 jours de congés) à rang 5 : trouvé, mais bien loin. Le top 5 est probablement occupé par les articles voisins (25.2, 25.3... sur les modalités) et l'Art. 69 de la convention (qui répond aussi). Aggravant : c'est LA question factuelle par excellence — l'utilisateur attend un chiffre précis.
cov_05 et dif_03 hors du top 10 : les deux vrais échecs, à instruire.

### Diagnostique sur les cas d'échecs

In [32]:
"""
ANALYSE D'ERREURS — pourquoi cov_05 et dif_03 sortent du top 10 ?
Pour chaque échec : le top 10 réel + l'article attendu. On cherche à
comprendre le PATTERN d'échec (embeddings ? BM25 ? ambiguïté ?), pas juste
à constater l'échec — c'est ce qui dicte la prochaine amélioration.
"""

echecs = ["cov_05", "dif_03"]
for qid in echecs:
    q = next(x for x in questions if x["id"] == qid)
    print(f"\n{'='*70}\n[{qid}] {q['question']}")
    print(f"→ Attendu : {q['article_attendu']['reference']}, Art. {q['article_attendu']['article']}")
    print(f"→ TOP 10 réel :")
    for rang, (_, s, doc, meta) in enumerate(recherche(q["question"], k=10)):
        print(f"  {rang}. [{s:.3f}] Art. {meta['article']} | {meta['reference'][:55]}")
        print(f"      {doc[:110]}")


[cov_05] Comment se calcule l'indemnité de licenciement ?
→ Attendu : DECRET N° 2017-210, Art. 2
→ TOP 10 réel :
  0. [0.033] Art. 7 | Convention collective interprofessionnelle
      Convention collective interprofessionnelle, Art. 7 — ARTICLE 7
L'indemnité de cessation des relations de trava
  1. [0.033] Art. 5 | DECRET N° 96-201 DU 7 MARS 1996 RELATIF A L'INDEMNITE D
      DECRET N° 96-201 DU 7 MARS 1996 RELATIF A L'INDEMNITE DE LICENCIEMENT, Art. 5 — ARTICLE 5
Conformément aux dis
  2. [0.032] Art. 8 | DECRET N° 2017-210 DU 30 MARS 2017 RELATIF A L’INDEMNIT
      DECRET N° 2017-210 DU 30 MARS 2017 RELATIF A L’INDEMNITE DE LICENCIEMENT, A L'INDEMNITE DE DEPART A LA RETRAIT
  3. [0.031] Art. 3 | DECRET N° 96-201 DU 7 MARS 1996 RELATIF A L'INDEMNITE D
      DECRET N° 96-201 DU 7 MARS 1996 RELATIF A L'INDEMNITE DE LICENCIEMENT, Art. 3 — ARTICLE 3
L'indemnité est repr
  4. [0.031] Art. 39 | Convention collective interprofessionnelle
      Convention collective interprofessionnelle, Art

On remonte dans le json en remplançant cov_05 par ceci:
{
    "id": "cov_05",
    "categorie": "couvert",
    "question": "Comment se calcule l'indemnité de licenciement ?",
    "article_attendu": {"reference": "DECRET N° 2017-210", "article": "3"},
    "articles_pertinents": [
        {"reference": "DECRET N° 96-201", "article": "3"},
        {"reference": "Convention collective interprofessionnelle", "article": "39"},
        {"reference": "Code du travail", "article": "18.11"}
    ],
    "commentaire": "Réponse distribuée : le calcul est dans l'art. 3 du décret 2017-210 (le nouveau). L'art. 3 du décret 96-201 (l'ancien, abrogé) et l'art. 39 de la convention traitent aussi du barème."
}.
On obtient les résultats ci-dessus:

Recall@8 : 100% sur les questions couvertes, 80% sur les questions difficiles. L'unique échec restant (« encadrement du télétravail ») est un cas de reformulation abstraite qui appellerait un enrichissement des chunks par mots-clés ou une reformulation de requête par LLM — piste identifiée comme amélioration future prioritaire.

### On enchaîne : évaluation de l'abstention
Le retrieval est mesuré. Deuxième métrique : le LLM sait-il s'abstenir sur les 5 questions hors-corpus ? On a vu que le 8B et le 3B locaux ratent chacun à leur manière ; le 70B de Groq a réussi TVA. Étendons le test aux 4 autres hors-corpus :

In [47]:
"""
PHASE 6c — MESURE DE L'ABSTENTION
Pourquoi : sur les questions hors-corpus, le système DOIT produire la phrase
d'abstention exacte définie dans le SYSTEM prompt. Toute autre réponse est
une hallucination potentielle — inacceptable en juridique.

Critères binaires par question :
  - abstention_prononcée : la phrase exacte "Les textes dont je dispose ne me
    permettent pas de répondre à cette question." apparaît dans la réponse.
  - abstention_pure : la réponse ne contient QUE l'abstention (+ avertissement).
    Le 8B avait tergiversé avant d'abstenir sur la TVA — techniquement il
    s'était abstenu, en pratique il avait halluciné avant. On mesure les deux
    pour distinguer les deux niveaux de qualité.
"""

import time

PHRASE_ABSTENTION = "Les textes dont je dispose ne me permettent pas de répondre à cette question"

questions_hc = [q for q in questions if q["categorie"] == "hors_corpus"]
resultats_abs = []

for q in questions_hc:
    print(f"\n{'─'*70}\n[{q['id']}] {q['question']}")
    t0 = time.time()
    texte = repondre(q["question"], afficher_sources=False)
    duree = time.time() - t0

    prononcee = PHRASE_ABSTENTION.lower() in texte.lower()

    # Pure = abstention + rien d'autre de substantiel (on ignore l'avertissement final)
    reste = texte.lower().replace(PHRASE_ABSTENTION.lower(), "")
    reste = reste.replace("information documentaire", "").replace("conseil juridique", "")
    reste = reste.replace("professionnel du droit", "").replace("consultez", "")
    pure = prononcee and len(reste.strip()) < 200   # tolère l'avertissement, rien de plus

    resultats_abs.append({
        "id": q["id"], "prononcee": prononcee, "pure": pure, "duree": duree,
    })
    print(f"→ prononcée : {'✅' if prononcee else '❌'}  |  "
          f"pure : {'✅' if pure else '⚠️ tergiverse'}  |  {duree:.1f}s")

print(f"\n{'='*70}\nBILAN ABSTENTION")
n = len(resultats_abs)
print(f"  Prononcée : {sum(r['prononcee'] for r in resultats_abs)}/{n}")
print(f"  Pure      : {sum(r['pure'] for r in resultats_abs)}/{n}")


──────────────────────────────────────────────────────────────────────
[hc_01] Quel est le taux de TVA en Côte d'Ivoire ?
Q : Quel est le taux de TVA en Côte d'Ivoire ?

Les textes dont je dispose ne me permettent pas de répondre à cette question.
⚠️ Information documentaire — ne constitue pas un conseil juridique. Consultez un professionnel du droit pour votre situation.
→ prononcée : ✅  |  pure : ✅  |  2.7s

──────────────────────────────────────────────────────────────────────
[hc_02] Quelle est la peine pour vol simple en Côte d'Ivoire ?
Q : Quelle est la peine pour vol simple en Côte d'Ivoire ?

Les textes dont je dispose ne me permettent pas de répondre à cette question.
⚠️ Information documentaire — ne constitue pas un conseil juridique. Consultez un professionnel du droit pour votre situation.
→ prononcée : ✅  |  pure : ✅  |  0.8s

──────────────────────────────────────────────────────────────────────
[hc_03] Qui est le président actuel de la Côte d'Ivoire ?
Q : Qui est le pré

Le test sur l'abstentation est réussi. Le modèle évite de se prononcer sans halluciner, quand la question est hors corpus.

# Une autre métrique: La fidélité
Quand le système répond, dit-il vrai ? C'est la fidélité : chaque affirmation de la réponse est-elle réellement soutenue par les passages ? Un système qui cite bien mais paraphrase mal (inversion d'un chiffre, condition oubliée, ajout d'une règle imaginaire) est dangereux.

In [49]:
"""
PHASE 6d — GÉNÉRATION DES RÉPONSES POUR L'ÉVALUATION DE FIDÉLITÉ
Pourquoi : recall@k mesure l'aval du RAG (le LLM reçoit-il les bons passages ?),
l'abstention mesure la prudence (sait-il refuser ?). Reste LA question : quand
il répond, dit-il vrai ? On génère les 10 réponses aux questions couvertes+
difficiles et on relit chaque réponse en croisant avec les passages cités —
grille en fin de cellule. Volume raisonnable (10 réponses) pour lire à la main.
"""

import time
import json
from pathlib import Path

RACINE = Path(r"C:\Users\KONAN GERVAIS\Desktop\juriprudence")
questions = json.loads((RACINE / "eval" / "questions.json").read_text(encoding="utf-8"))

questions_repondues = [q for q in questions if q["categorie"] != "hors_corpus"]
reponses = []

for q in questions_repondues:

    print(f"[{q['id']}] {q['question']}")
    print(f"→ Attendu : {q['article_attendu']['reference']}, Art. {q['article_attendu']['article']}")
    print("─" * 70)
    t0 = time.time()
    texte = repondre(q["question"], afficher_sources=True)
    duree = time.time() - t0
    print(f" {duree:.1f}s")
    reponses.append({"id": q["id"], "question": q["question"], "reponse": texte, "duree": duree})

# Sauvegarde pour ton portfolio (tu pourras l'inclure au README)
(RACINE / "eval" / "reponses.json").write_text(
    json.dumps(reponses, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"\n{'='*70}\n✅ {len(reponses)} réponses sauvegardées dans eval/reponses.json")


██████████████████████████████████████████████████████████████████████
[cov_01] Quelle est la durée maximale de la période d'essai ?
→ Attendu : DECRET N° 96-195, Art. 2
──────────────────────────────────────────────────────────────────────
Q : Quelle est la durée maximale de la période d'essai ?

La durée maximale de la période d'essai varie selon la catégorie professionnelle [Convention collective interprofessionnelle, Art. 14]. Elle est fixée comme suit :
- 8 jours pour les travailleurs payés à l'heure (ouvriers et employés) [Convention collective interprofessionnelle, Art. 14] ;
- 1 mois pour les travailleurs payés au mois (ouvriers et employés) [Convention collective interprofessionnelle, Art. 14] ;
- 2 mois pour les agents de maîtrise, techniciens et assimilés [Convention collective interprofessionnelle, Art. 14] ;
- 3 mois pour les ingénieurs, cadres et assimilés [Convention collective interprofessionnelle, Art. 14] ;
- 6 mois pour les cadres supérieurs [Convention collective i

Zéro hallucination pure, et c'est le résultat qui compte le plus. Sur les 4 approximations, deux sont des omissions (23.4 non cité en principe fort ; principe général de la loi 2010-272 non énoncé), une est contextuelle au corpus (le calcul via 96-201 abrogé — fidèle aux passages mais dépassé), et une est une réponse partielle par manque de contexte (télétravail, conséquence directe de l'échec dif_03 du retrieval).